In [55]:
# ------------------------------------------------------------
# 📦 Imports & Configuration
# ------------------------------------------------------------
import os
import cv2
import time
import torch
import numpy as np
import pandas as pd
from collections import deque, defaultdict
from ultralytics import YOLO
from sklearn.metrics import average_precision_score

# ------------------------------------------------------------
# JSON STYLE CONFIG
# ------------------------------------------------------------
CONFIG = {
    "VIDEO_PATH": "/home1/jtt_1/mtp/okutama-action/TrainSetVideos/Drone2/Morning/2.1.1.mp4",

    # Models
    "YOLO_MODEL_PATH": "/home1/jtt_1/mtp/models-2/yolo-person-model.pt",
    "TSF_MODEL_PATH": "/home1/jtt_1/mtp/timesformer_base_patch16_224_k400",
    "TSF_CHECKPOINT": "/home1/jtt_1/mtp/models-2/best_timesformer_epoch68.pt",

    # Training CSV (for collecting all action classes)
    "GT_BBOX_CSV": "/home1/jtt_1/mtp/timesformer-person-dataset/train_person_labels.csv",

    # Hyperparameters
    "NUM_FRAMES": 16,
    "TSF_INPUT_SIZE": 224,
    "IOU_THRESH": 0.1,

    # # Detection settings
    "CONF_THRESH": 0.5,
    "IMG_SIZE": 1280,  # for YOLOv8 inference (keep as int, not tuple)

    # Test settings
    "TEST_MODE": False,
    "TEST_MAX_FRAMES": 10,

    # Device
    "DEVICE": "cuda" if torch.cuda.is_available() else "cpu",
    }

# ------------------------------------------------------------
# AUTO FIELDS (computed)
# ------------------------------------------------------------
CONFIG["VIDEO_NAME"] = os.path.splitext(os.path.basename(CONFIG["VIDEO_PATH"]))[0]
CONFIG["OUTPUT_DIR"] = f"/home1/jtt_1/mtp/full-outputs-without-tracking/{CONFIG['VIDEO_NAME']}"
os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)

CONFIG["RESULTS_CSV"] = f"{CONFIG['OUTPUT_DIR']}/{CONFIG['VIDEO_NAME']}_tracking.csv"

print("[INFO] VIDEO_NAME   :", CONFIG["VIDEO_NAME"])
print("[INFO] OUTPUT_DIR   :", CONFIG["OUTPUT_DIR"])
print("[INFO] RESULTS_CSV  :", CONFIG["RESULTS_CSV"])


[INFO] VIDEO_NAME   : 2.1.1
[INFO] OUTPUT_DIR   : /home1/jtt_1/mtp/full-outputs-without-tracking/2.1.1
[INFO] RESULTS_CSV  : /home1/jtt_1/mtp/full-outputs-without-tracking/2.1.1/2.1.1_tracking.csv


In [56]:
# ------------------------------------------------------------
# Load unique ACTION_CLASSES from GT CSV
# ------------------------------------------------------------
df = pd.read_csv(CONFIG["GT_BBOX_CSV"])

unique_actions = set()
for acts in df["Actions"]:
    if pd.isna(acts):
        continue
    for a in str(acts).split(","):
        unique_actions.add(a.strip())

CONFIG["ACTION_CLASSES"] = sorted(unique_actions)
CONFIG["NUM_CLASSES"] = len(CONFIG["ACTION_CLASSES"])

print("[INFO] Loaded ACTION_CLASSES:", CONFIG["ACTION_CLASSES"])
print("[INFO] NUM_CLASSES:", CONFIG["NUM_CLASSES"])
print("[INFO] Device:", CONFIG["DEVICE"])


[INFO] Loaded ACTION_CLASSES: ['Calling', 'Carrying', 'Drinking', 'Handshaking', 'Hugging', 'Lying', 'Pushing or Pulling', 'Reading', 'Running', 'Sitting', 'Standing', 'Walking', 'nan']
[INFO] NUM_CLASSES: 13
[INFO] Device: cuda


In [57]:
# ------------------------------------------------------------
# 🔍 ROBUST GT FINDER (OKUTAMA DATASET)
# ------------------------------------------------------------

import os

def find_gt_file(video_path, gt_root_dir):
    """
    Finds GT file by exact filename match across all resolution folders
    """

    video_name = os.path.splitext(os.path.basename(video_path))[0]

    for root, dirs, files in os.walk(gt_root_dir):
        for file in files:
            if file.endswith(".txt"):
                if os.path.splitext(file)[0] == video_name:
                    return os.path.join(root, file)

    return None

In [58]:
GT_ROOT = "/home1/jtt_1/mtp/okutama-action/TrainSetVideos/Labels/SingleActionLabels"

GT_FILE = find_gt_file(CONFIG["VIDEO_PATH"], GT_ROOT)

if GT_FILE is None:
    raise FileNotFoundError(f"No GT found for {CONFIG['VIDEO_PATH']}")

print("[INFO] GT FOUND:", GT_FILE)

[INFO] GT FOUND: /home1/jtt_1/mtp/okutama-action/TrainSetVideos/Labels/SingleActionLabels/3840x2160/2.1.1.txt


In [59]:
# ------------------------------------------------------------
# 📦 LOAD GT (TXT FORMAT)
# ------------------------------------------------------------

from collections import defaultdict

gt_by_frame = defaultdict(list)

with open(GT_FILE, "r") as f:
    for line in f:

        parts = line.strip().split()

        if len(parts) < 10:
            continue  # skip bad lines

        track_id = int(parts[0])
        xmin = float(parts[1])
        ymin = float(parts[2])
        xmax = float(parts[3])
        ymax = float(parts[4])
        frame_id = int(parts[5])

        # Multi-label actions (important)
        actions = parts[9:]
        actions = [a.replace('"', '') for a in actions if a.strip() != ""]
        action = ",".join(actions)

        gt_by_frame[frame_id].append({
            "track_id": track_id,
            "bbox": [xmin, ymin, xmax, ymax],
            "action": action
        })

print(f"[INFO] Loaded GT for {len(gt_by_frame)} frames.")

[INFO] Loaded GT for 1252 frames.


In [60]:
cap = cv2.VideoCapture(CONFIG["VIDEO_PATH"])

if not cap.isOpened():
    print(f"[ERROR] Could not open video file: {CONFIG['VIDEO_PATH']}")
else:
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration_sec = total_frames / fps if fps > 0 else 0

    print(f"[INFO] Video Name       : {CONFIG['VIDEO_NAME']}")
    print(f"[INFO] Total Frames     : {total_frames}")
    print(f"[INFO] FPS              : {fps:.2f}")
    print(f"[INFO] Duration (secs)  : {duration_sec:.2f}")
    print(f"[INFO] Duration (mins)  : {duration_sec/60:.2f}")

cap.release()

[INFO] Video Name       : 2.1.1
[INFO] Total Frames     : 1235
[INFO] FPS              : 29.97
[INFO] Duration (secs)  : 41.21
[INFO] Duration (mins)  : 0.69


In [61]:
# ------------------------------------------------------------
# 🧠 LOAD MODELS (FINAL WORKING VERSION)
# ------------------------------------------------------------
print("[INFO] Loading YOLO model...")
yolo = YOLO(CONFIG["YOLO_MODEL_PATH"])
print("[INFO] YOLO loaded successfully.")


# ------------------------------------------------------------
# Load TimeSformer with correct number of action classes
# ------------------------------------------------------------
print("[INFO] Loading TimeSformer model...")

from transformers import AutoImageProcessor, TimesformerForVideoClassification
import torch.nn as nn

# Load action class list
action_labels = CONFIG["ACTION_CLASSES"]
num_actions = CONFIG["NUM_CLASSES"]

print(f"[INFO] TimeSformer will use {num_actions} action classes:")
print(action_labels)


# 1. Load processor from BASE MODEL DIRECTORY
processor = AutoImageProcessor.from_pretrained(CONFIG["TSF_MODEL_PATH"])


# 2. Load base model architecture
tsf_model = TimesformerForVideoClassification.from_pretrained(CONFIG["TSF_MODEL_PATH"])


# 3. Replace classification head
tsf_model.classifier = nn.Linear(tsf_model.classifier.in_features, num_actions)


# 4. Load your fine-tuned checkpoint (.pt file)
state_dict = torch.load(CONFIG["TSF_CHECKPOINT"], map_location=CONFIG["DEVICE"])
tsf_model.load_state_dict(state_dict, strict=True)


# 5. Move model to device
tsf_model = tsf_model.to(CONFIG["DEVICE"]).eval()

print("[INFO] TimeSformer loaded successfully.")
print("-------------------------------------")


[INFO] Loading YOLO model...
[INFO] YOLO loaded successfully.
[INFO] Loading TimeSformer model...
[INFO] TimeSformer will use 13 action classes:
['Calling', 'Carrying', 'Drinking', 'Handshaking', 'Hugging', 'Lying', 'Pushing or Pulling', 'Reading', 'Running', 'Sitting', 'Standing', 'Walking', 'nan']


[INFO] TimeSformer loaded successfully.
-------------------------------------


In [62]:
# ------------------------------------------------------------
# 🔧 YOLO OUTPUT → BBOX LIST
# ------------------------------------------------------------

def yolo_to_bboxes(yolo_out):
    """
    Converts YOLO output to list of bounding boxes
    format: [x1, y1, x2, y2]
    """

    if len(yolo_out) == 0:
        return []

    result = yolo_out[0]

    if result.boxes is None:
        return []

    boxes = result.boxes.xyxy.cpu().numpy()

    bboxes = []
    for box in boxes:
        x1, y1, x2, y2 = box[:4]
        bboxes.append([x1, y1, x2, y2])

    return bboxes

In [63]:
# ------------------------------------------------------------
# 🔧 CROP FUNCTION
# ------------------------------------------------------------

def crop_box(frame, bbox):
    x1, y1, x2, y2 = map(int, bbox)

    h, w = frame.shape[:2]

    x1 = max(0, min(x1, w-1))
    y1 = max(0, min(y1, h-1))
    x2 = max(0, min(x2, w-1))
    y2 = max(0, min(y2, h-1))

    if x2 <= x1 or y2 <= y1:
        return None

    return frame[y1:y2, x1:x2]

In [64]:
# ------------------------------------------------------------
# 📊 ULTRA VISUAL SYSTEM OVERLAY (DASHBOARD STYLE)
# ------------------------------------------------------------

import psutil
import cv2
import numpy as np

def draw_overlay(frame, frame_idx, total_frames, fps_proc, num_detections):

    h, w = frame.shape[:2]

    # --------------------------------------------------
    # 📊 SYSTEM STATS
    # --------------------------------------------------
    cpu = psutil.cpu_percent()
    mem = psutil.virtual_memory().percent

    # Text lines (clean + aligned)
    lines = [
        f"Frame   : {frame_idx}/{total_frames}",
        f"FPS     : {fps_proc:.2f}",
        f"CPU     : {cpu:.1f}%",
        f"RAM     : {mem:.1f}%",
        f"Objects : {num_detections}"
    ]

    # --------------------------------------------------
    # 🎨 STYLE CONFIG
    # --------------------------------------------------
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.55
    thickness = 1
    line_height = 22
    padding = 12
    radius = 10

    # Compute width
    max_width = 0
    for line in lines:
        (w_text, _), _ = cv2.getTextSize(line, font, font_scale, thickness)
        max_width = max(max_width, w_text)

    box_w = max_width + padding * 2
    box_h = line_height * len(lines) + padding

    x1, y1 = 10, 10
    x2, y2 = x1 + box_w, y1 + box_h

    # --------------------------------------------------
    # 🌫 GLASS BACKGROUND (TRANSPARENT)
    # --------------------------------------------------
    overlay = frame.copy()

    # Rounded rectangle (manual)
    cv2.rectangle(overlay, (x1+radius, y1), (x2-radius, y2), (30, 30, 30), -1)
    cv2.rectangle(overlay, (x1, y1+radius), (x2, y2-radius), (30, 30, 30), -1)

    cv2.circle(overlay, (x1+radius, y1+radius), radius, (30,30,30), -1)
    cv2.circle(overlay, (x2-radius, y1+radius), radius, (30,30,30), -1)
    cv2.circle(overlay, (x1+radius, y2-radius), radius, (30,30,30), -1)
    cv2.circle(overlay, (x2-radius, y2-radius), radius, (30,30,30), -1)

    alpha = 0.5
    frame = cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0)

    # --------------------------------------------------
    # 🟢 ACCENT BAR (LEFT)
    # --------------------------------------------------
    cv2.rectangle(frame, (x1, y1), (x1+4, y2), (0, 255, 150), -1)

    # --------------------------------------------------
    # 🔤 DRAW TEXT WITH SHADOW
    # --------------------------------------------------
    for i, line in enumerate(lines):
        y = y1 + padding + (i + 1) * line_height - 6

        # Shadow
        cv2.putText(frame, line, (x1 + 10, y+1),
                    font, font_scale, (0,0,0), thickness+1, cv2.LINE_AA)

        # Main text
        cv2.putText(frame, line, (x1 + 10, y),
                    font, font_scale, (220, 255, 220), thickness, cv2.LINE_AA)

    return frame

In [65]:
# ------------------------------------------------------------
# 🎨 ULTRA VISUAL DETECTION DRAW (RESEARCH LEVEL)
# ------------------------------------------------------------

import cv2
import numpy as np
import random

def generate_class_colors(class_names):
    random.seed(42)
    colors = {}
    for cls in class_names:
        colors[cls] = tuple(random.randint(50, 255) for _ in range(3))  # avoid dark colors
    return colors

CLASS_COLORS = generate_class_colors(CONFIG["ACTION_CLASSES"])

# ------------------------------------------------------------
# 🎨 FINAL VISUAL DETECTION DRAW (CLEAN + STABLE)
# ------------------------------------------------------------

def draw_detection(frame, bbox, labels, scores=None):

    x1, y1, x2, y2 = map(int, bbox)

    # Safety check
    if labels is None or labels == "":
        return frame

    # Handle labels
    if isinstance(labels, str):
        labels = [l.strip() for l in labels.split(",") if l.strip() != ""]

    if len(labels) == 0:
        return frame

    # Color
    color = CLASS_COLORS.get(labels[0], (0, 255, 0))

    # --------------------------------------------------
    # 🟩 BOUNDING BOX (with border)
    # --------------------------------------------------
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2, cv2.LINE_AA)

    # Outer border for contrast
    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 0), 1, cv2.LINE_AA)

    # --------------------------------------------------
    # 📝 TEXT PREP
    # --------------------------------------------------
    text_lines = []
    for i, lbl in enumerate(labels):
        if scores is not None and i < len(scores):
            text_lines.append(f"{lbl}: {scores[i]:.2f}")
        else:
            text_lines.append(lbl)

    # Font config
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.5
    thickness = 1
    line_height = 18
    padding = 6

    # Compute width
    max_width = 0
    for txt in text_lines:
        (w, _), _ = cv2.getTextSize(txt, font, font_scale, thickness)
        max_width = max(max_width, w)

    box_w = max_width + padding * 2
    box_h = line_height * len(text_lines) + padding

    # --------------------------------------------------
    # 📍 SMART POSITION (above or below bbox)
    # --------------------------------------------------
    if y1 - box_h - 5 > 0:
        # draw above
        y_text = y1 - box_h - 5
    else:
        # draw below
        y_text = y2 + 5

    x_text = x1

    # Clamp inside frame
    h, w = frame.shape[:2]
    x_text = max(0, min(x_text, w - box_w))
    y_text = max(0, min(y_text, h - box_h))

    # --------------------------------------------------
    # 🌈 SEMI-TRANSPARENT BACKGROUND (SAFE)
    # --------------------------------------------------
    sub_img = frame[y_text:y_text+box_h, x_text:x_text+box_w]
    overlay = sub_img.copy()

    cv2.rectangle(
        overlay,
        (0, 0),
        (box_w, box_h),
        color,
        -1
    )

    alpha = 0.7
    frame[y_text:y_text+box_h, x_text:x_text+box_w] = cv2.addWeighted(
        overlay, alpha, sub_img, 1 - alpha, 0
    )

    # --------------------------------------------------
    # 🔤 TEXT COLOR (AUTO CONTRAST)
    # --------------------------------------------------
    brightness = (color[0]*0.299 + color[1]*0.587 + color[2]*0.114)
    text_color = (0, 0, 0) if brightness > 140 else (255, 255, 255)

    # --------------------------------------------------
    # ✨ DRAW TEXT (WITH SHADOW)
    # --------------------------------------------------
    for i, txt in enumerate(text_lines):

        y_offset = y_text + padding + (i + 1) * line_height - 6

        # # Shadow
        # cv2.putText(
        #     frame,
        #     txt,
        #     (x_text + padding + 1, y_offset + 1),
        #     font,
        #     font_scale,
        #     (0, 0, 0),
        #     thickness + 1,
        #     cv2.LINE_AA
        # )

        # Main text
        cv2.putText(
            frame,
            txt,
            (x_text + padding, y_offset),
            font,
            font_scale,
            text_color,
            thickness,
            cv2.LINE_AA
        )

    return frame

In [ ]:
# ------------------------------------------------------------
# 🔥 CELL 4 — INFERENCE (NO TRACKING + VIDEO + PROGRESS)
# ------------------------------------------------------------

print("[INFO] Starting inference (NO TRACKING)...")

results_rows = []

cap = cv2.VideoCapture(CONFIG["VIDEO_PATH"])
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

output_video_path = f"{CONFIG['OUTPUT_DIR']}/{CONFIG['VIDEO_NAME']}_output.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
video_writer = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

print("[INFO] Saving video to:", output_video_path)

frame_idx = 0
start_time = time.time()

while True:

    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx >= total_frames:
        break

    if CONFIG["TEST_MODE"] and frame_idx >= CONFIG["TEST_MAX_FRAMES"]:
        print(f"[INFO] TEST MODE: Stopping at {CONFIG['TEST_MAX_FRAMES']} frames")
        break

    # ------------------------------------------------------
    # YOLO DETECTION
    # ------------------------------------------------------
    yolo_out = yolo.predict(frame, conf=CONFIG["CONF_THRESH"], imgsz=CONFIG["IMG_SIZE"], verbose=False)
    det_bboxes = yolo_to_bboxes(yolo_out)

    for bbox in det_bboxes:

        x1, y1, x2, y2 = map(int, bbox)
        crop = crop_box(frame, bbox)

        if crop is None or crop.size == 0:
            continue

        # --------------------------------------------------
        # FAKE CLIP (repeat frame)
        # --------------------------------------------------
        resized = cv2.resize(crop, (CONFIG["TSF_INPUT_SIZE"], CONFIG["TSF_INPUT_SIZE"]))
        clip_np = np.stack([resized]*CONFIG["NUM_FRAMES"], axis=0).astype(np.uint8)

        inputs = processor(list(clip_np), return_tensors="pt")
        inputs = {k: v.to(CONFIG["DEVICE"]) for k, v in inputs.items()}

        with torch.no_grad():
            logits = tsf_model(**inputs).logits[0]
            probs = torch.sigmoid(logits).cpu().numpy()

        # best_idx = probs.argmax()
        # best_label = action_labels[best_idx]
        # best_score = float(probs[best_idx])
        # --------------------------------------------------
        # MULTI-LABEL PREDICTION
        # --------------------------------------------------

        threshold = 0.8   # you can tune this

        selected_indices = np.where(probs >= threshold)[0]

        if len(selected_indices) == 0:
            # fallback to best
            best_idx = probs.argmax()
            selected_indices = [best_idx]

        labels = [action_labels[i] for i in selected_indices]
        scores = [float(probs[i]) for i in selected_indices]

        # Convert to string
        best_label = ",".join(labels)
        best_score = max(scores)

        # Save row
        results_rows.append([
            frame_idx, -1,
            x1, y1, x2, y2,
            best_label, best_score
        ])

        # Draw detections
        frame = draw_detection(frame, bbox, labels, scores)

        # Compute FPS
        elapsed = time.time() - start_time
        fps_proc = frame_idx / elapsed if elapsed > 0 else 0

        # Draw overlay
        frame = draw_overlay(
            frame,
            frame_idx,
            total_frames,
            fps_proc,
            len(det_bboxes)
        )

    video_writer.write(frame)

    # ------------------------------------------------------
    # 📊 PROGRESS DISPLAY
    # ------------------------------------------------------
    if frame_idx % 10 == 0:   # update every 10 frames (avoid slowdown)

        elapsed = time.time() - start_time
        fps_proc = frame_idx / elapsed if elapsed > 0 else 0

        progress = (frame_idx / total_frames) * 100 if total_frames > 0 else 0

        remaining_frames = total_frames - frame_idx
        eta = remaining_frames / fps_proc if fps_proc > 0 else 0

        print(f"[PROGRESS] Frame {frame_idx}/{total_frames} "
              f"({progress:.2f}%) | FPS: {fps_proc:.2f} | ETA: {eta:.1f}s")

    frame_idx += 1

cap.release()
video_writer.release()

print("[INFO] Inference done.")

[INFO] Starting inference (NO TRACKING)...
[INFO] Saving video to: /home1/jtt_1/mtp/full-outputs-without-tracking/2.1.1/2.1.1_output.mp4
[PROGRESS] Frame 0/1235 (0.00%) | FPS: 0.00 | ETA: 0.0s
[PROGRESS] Frame 10/1235 (0.81%) | FPS: 0.43 | ETA: 2847.0s
[PROGRESS] Frame 20/1235 (1.62%) | FPS: 0.41 | ETA: 2954.0s
[PROGRESS] Frame 30/1235 (2.43%) | FPS: 0.40 | ETA: 3049.8s
[PROGRESS] Frame 40/1235 (3.24%) | FPS: 0.39 | ETA: 3069.8s
[PROGRESS] Frame 50/1235 (4.05%) | FPS: 0.39 | ETA: 3059.2s
[PROGRESS] Frame 60/1235 (4.86%) | FPS: 0.39 | ETA: 3013.4s
[PROGRESS] Frame 70/1235 (5.67%) | FPS: 0.38 | ETA: 3042.3s
[PROGRESS] Frame 80/1235 (6.48%) | FPS: 0.39 | ETA: 2982.3s


In [ ]:
# ---------------------------
# Cell 5: Save inference results to CSV
# ---------------------------
import pandas as pd
cols = ["frame", "track_id", "xmin", "ymin", "xmax", "ymax", "action", "score"]
pred_df = pd.DataFrame(results_rows, columns=cols)

# Save CSV
pred_csv_path = CONFIG.get("RESULTS_CSV", f"{CONFIG['OUTPUT_DIR']}/{CONFIG['VIDEO_NAME']}_predictions.csv")
pred_df.to_csv(pred_csv_path, index=False)
print(f"[INFO] Saved predictions to: {pred_csv_path}")


[INFO] Saved predictions to: /home1/jtt_1/mtp/full-outputs-without-tracking/2.1.1/2.1.1_tracking.csv


In [ ]:
# GT_FILE, pred_csv_path

In [ ]:
import pandas as pd
import numpy as np
import cv2
from collections import defaultdict

# pred_csv_path = "/home1/jtt_1/mtp/full-outputs/2.1.1/2.1.1_tracking.csv"
# GT_FILE = "/home1/jtt_1/mtp/okutama-action/TrainSetVideos/Labels/SingleActionLabels/3840x2160/2.1.1.txt"  # change this

IOU_THRESH = 0.5
SAVE_DEBUG = True
# CONFIG["OUTPUT_DIR"]

In [ ]:
def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    inter = max(0, xB - xA) * max(0, yB - yA)

    areaA = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    areaB = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])

    union = areaA + areaB - inter
    return inter / union if union > 0 else 0

In [ ]:
# Load predictions
pred_df = pd.read_csv(pred_csv_path)

def clean_action(x):
    if isinstance(x, str):
        return x.split(",")[0].strip()
    return x

pred_df["action"] = pred_df["action"].apply(clean_action)

# Load GT
gt_data = []
with open(GT_FILE, "r") as f:
    for line in f:
        parts = line.strip().split()

        frame = int(parts[0])
        xmin, ymin, xmax, ymax = map(int, parts[1:5])
        person_label = parts[-2].replace('"', '')
        action_label = parts[-1].replace('"', '')

        action = action_label   # ignore "Person"

        gt_data.append([frame, xmin, ymin, xmax, ymax, action])

gt_df = pd.DataFrame(gt_data, columns=[
    "frame", "xmin", "ymin", "xmax", "ymax", "action"
])

# 🔥 REMOVE DUPLICATES (critical)
gt_df = gt_df.drop_duplicates(subset=["frame","xmin","ymin","xmax","ymax"])

In [ ]:
if CONFIG["TEST_MODE"]:
    gt_df = gt_df[gt_df["frame"] < CONFIG["TEST_MAX_FRAMES"]]

In [ ]:
# ------------------------------------------------------------
# REMOVE DUPLICATE GT (CRITICAL)
# ------------------------------------------------------------
gt_df = gt_df.drop_duplicates(
    subset=["frame","xmin","ymin","xmax","ymax"]
)

# ------------------------------------------------------------
# METRICS INIT
# ------------------------------------------------------------
TP, FP, FN = 0, 0, 0
correct_action = 0

for frame in sorted(gt_df["frame"].unique()):

    gt_frame = gt_df[gt_df.frame == frame]
    pred_frame = pred_df[pred_df.frame == frame]

    used_gt = set()

    for _, pred in pred_frame.iterrows():

        pred_box = [pred.xmin, pred.ymin, pred.xmax, pred.ymax]
        pred_actions = [a.strip() for a in str(pred.action).split(",")]

        best_iou = 0
        best_gt_idx = None

        for idx, gt in gt_frame.iterrows():

            if idx in used_gt:
                continue

            gt_box = [gt.xmin, gt.ymin, gt.xmax, gt.ymax]
            iou = compute_iou(pred_box, gt_box)

            if iou > best_iou:
                best_iou = iou
                best_gt_idx = idx

        # ----------------------------
        # MATCH FOUND
        # ----------------------------
        if best_iou >= IOU_THRESH:

            TP += 1
            used_gt.add(best_gt_idx)

            gt_action = gt_frame.loc[best_gt_idx].action

            # ✅ single-label correctness
            if gt_action in pred_actions:
                correct_action += 1

        else:
            FP += 1

    # ----------------------------
    # UNMATCHED GT → FN
    # ----------------------------
    FN += len(gt_frame) - len(used_gt)

# ------------------------------------------------------------
# METRICS
# ------------------------------------------------------------
precision = TP / (TP + FP + 1e-6)
recall = TP / (TP + FN + 1e-6)
action_acc = correct_action / (TP + 1e-6)

# ------------------------------------------------------------
# PRINT
# ------------------------------------------------------------
print("\n===== SINGLE-LABEL EVALUATION =====")
print(f"TP: {TP}, FP: {FP}, FN: {FN}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"Action Accuracy: {action_acc:.4f}")


===== SINGLE-LABEL EVALUATION =====
TP: 5, FP: 73, FN: 809
Precision: 0.0641
Recall: 0.0061
Action Accuracy: 0.4000


In [ ]:
print("GT unique frames:", len(gt_df["frame"].unique()))
print("Pred unique frames:", len(pred_df["frame"].unique()))

GT unique frames: 10
Pred unique frames: 10


In [ ]:
print("GT frames:", gt_df["frame"].min(), "to", gt_df["frame"].max())
print("Pred frames:", pred_df["frame"].min(), "to", pred_df["frame"].max())

GT frames: 0 to 9
Pred frames: 0 to 9
